# Objective
# Our core research question:
    Can we accurately predict apartment prices in Buenos Aires based on characteristics like size, type, and location?
    Answering that question is a three-stage process, all of which this lesson covers:
    Clean, structured data — raw data is rarely machine-learning-ready straight from the source.
    Thoughtful feature engineering — transforming raw columns into mathematical inputs a model can actually use.
    A proper train-test split — ensuring our model is evaluated on data it has never seen during training.

# Project workflow
    Used glob to locate multiple CSV files matching a naming pattern and load them into a single DataFrame
    Filtered data to a homogeneous market segment using .loc[] with lambda functions and .query()
    Removed outliers systematically using quantile-based bounds that adapt to any data distribution
    Extracted numeric and categorical features from raw string columns
    Diagnosed missing-value patterns using bar charts and the missingno matrix
    Detected and resolve multicollinearity between numeric features using a correlation heatmap
    Understood why one-hot encoding is needed for categorical variables, why category_encoders is preferred over scikit-learn's OneHotEncoder, and how to apply it
    Split data into training and test sets with scikit-learn's train_test_split

In [1]:
import numpy as np
import pandas as pd
from glob import glob

# Import and Merge Multiple CSV Files

In [2]:
datasets=glob("buenos-aires-real-estate-*.csv")
datasets

['buenos-aires-real-estate-1.csv',
 'buenos-aires-real-estate-2.csv',
 'buenos-aires-real-estate-3.csv',
 'buenos-aires-real-estate-4.csv',
 'buenos-aires-real-estate-5.csv']

In [3]:
files=[pd.read_csv(x) for x in datasets]

In [6]:
df=pd.concat(files, ignore_index=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43029 entries, 0 to 43028
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   operation                   43029 non-null  str    
 1   property_type               43029 non-null  str    
 2   place_with_parent_names     43029 non-null  str    
 3   lat-lon                     34734 non-null  str    
 4   price                       38073 non-null  float64
 5   currency                    38072 non-null  str    
 6   price_aprox_local_currency  38073 non-null  float64
 7   price_aprox_usd             38073 non-null  float64
 8   surface_total_in_m2         29871 non-null  float64
 9   surface_covered_in_m2       36420 non-null  float64
 10  price_usd_per_m2            24449 non-null  float64
 11  price_per_m2                32642 non-null  float64
 12  floor                       6505 non-null   float64
 13  rooms                       23806 non-null

### We will wrap this into a function called merged_files so that any future notebook can call it with a single line and get back the fully merged DataFrame.

In [8]:
def merged_files(data):
    return pd.concat([pd.read_csv(file) for file in glob(data)], ignore_index=True)
merged_files("buenos-aires-real-estate-*.csv").info()

<class 'pandas.DataFrame'>
RangeIndex: 43029 entries, 0 to 43028
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   operation                   43029 non-null  str    
 1   property_type               43029 non-null  str    
 2   place_with_parent_names     43029 non-null  str    
 3   lat-lon                     34734 non-null  str    
 4   price                       38073 non-null  float64
 5   currency                    38072 non-null  str    
 6   price_aprox_local_currency  38073 non-null  float64
 7   price_aprox_usd             38073 non-null  float64
 8   surface_total_in_m2         29871 non-null  float64
 9   surface_covered_in_m2       36420 non-null  float64
 10  price_usd_per_m2            24449 non-null  float64
 11  price_per_m2                32642 non-null  float64
 12  floor                       6505 non-null   float64
 13  rooms                       23806 non-null